# 04 — Probability Calibration, Cost-based Threshold & PDO Credit Scoring

**Mục đích:** Báo cáo thực nghiệm và trực quan hóa toàn bộ quy trình:
1. **Probability Calibration trên tập Valid:** So sánh Platt scaling (fit tay bằng hồi quy logistic 1 chiều trên raw margin / output_margin) và Isotonic regression theo chỉ số Brier score, Log Loss, ROC-AUC và ECE.
2. **Tối ưu hóa ngưỡng theo ma trận chi phí:** Xác định ngưỡng cắt xác suất tối ưu ($p^*$) theo chi phí kinh doanh (FN = LGD × EAD với LGD = 0.45; FP = 5% × EAD) kèm phân tích độ nhạy với tỷ lệ chi phí FN:FP trong khoảng 3:1 đến 10:1.
3. **Quy đổi PD sang điểm tín dụng (PDO):** Chuyển đổi xác suất vỡ nợ đã hiệu chuẩn sang thang điểm chuẩn ngân hàng [300, 850] với Base Score = 600 tại odds 50:1 và PDO = 20, xác định ngưỡng điểm cắt (cutoff score).

**Phạm vi & Nguyên tắc kỹ thuật (project_plan.md mục 3.5 & 4.3):**
- **Dữ liệu:** Mô hình được huấn luyện trên tập **Train (18.000 dòng)**; toàn bộ bước hiệu chuẩn, chọn ngưỡng và phân tích độ nhạy được thực hiện trên tập **Valid (6.000 dòng)**.
- **Tập Test (6.000 dòng) được đóng băng hoàn toàn**, chỉ chạy một lần duy nhất ở Tuần 3 – T3 để báo cáo kết quả nghiệm thu cuối cùng.
- **Mô hình khảo sát:** Ứng viên chính `LightGBM tuned + Monotonic`, ứng viên đối chứng `CatBoost tuned`, và mô hình cơ sở `Logistic Regression (baseline)`.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.calibrate import (
    CalibratedPipeline,
    compare_calibrations_on_valid,
    evaluate_calibration,
    predict_margin,
)
from src.data_split import load_split_data
from src.evaluate import (
    compute_confusion_matrix_metrics,
    compute_expected_cost,
    find_optimal_threshold,
    sensitivity_analysis_cost_ratios,
)
from src.features import build_features
from src.pipelines import make_pipeline
from src.scoring import pd_to_score
from src.tune import fit_tuned_pipeline
from sklearn.linear_model import LogisticRegression

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 15)
pd.set_option("display.float_format", lambda x: f"{x:.5f}")
print("Setup completed successfully!")

## 1. Nạp Dữ liệu & Huấn luyện Mô hình trên tập Train

Tuân thủ quy tắc chống rò rỉ dữ liệu (No Data Leakage):
- Pipeline tiền xử lý và mô hình học hoàn toàn trên **Train (18.000 mẫu)**.
- Tập **Valid (6.000 mẫu)** chỉ dùng để trích xuất margin, fit calibrator và tìm ngưỡng chi phí.

In [ ]:
train_df, val_df, _ = load_split_data()
X_train, y_train = build_features(train_df)
X_val, y_val = build_features(val_df)
ead_val = np.maximum(val_df["BILL_AMT1"].to_numpy(dtype=float), 0.0)

print(f"Train shape: {X_train.shape}, Default rate: {y_train.mean():.4f}")
print(f"Valid shape: {X_val.shape}, Default rate: {y_val.mean():.4f}")

# Huấn luyện các mô hình trên tập Train (dùng siêu tham số Optuna đã chốt)
models = {
    "LightGBM tuned + Monotonic": fit_tuned_pipeline("lightgbm_monotone", X_train, y_train),
    "CatBoost tuned": fit_tuned_pipeline("catboost", X_train, y_train),
    "Logistic baseline": make_pipeline(LogisticRegression(max_iter=1000, random_state=42)).fit(X_train, y_train),
}
print("All candidate models fitted on Train!")

## 2. So sánh Hiệu chuẩn: Platt Scaling vs. Isotonic Regression

**Nguyên lý triển khai Platt scaling fit tay trên margin (project_plan.md mục 3.5e):**
- Với mô hình cây (LightGBM, CatBoost), TreeExplainer bóc tách đầu ra trên thang log-odds thô ($z = \text{margin}$). Khi áp dụng hàm Platt scaling $PD = \sigma(a \cdot z + b)$, mọi giá trị SHAP $\phi_i$ chỉ cần nhân với hệ số dốc $a$ để quy về thang logit của PD đã hiệu chuẩn: $\text{logit}(PD) = (a\phi_0 + b) + \sum a\phi_i$. Tổng này bảo toàn chính xác tính cộng tính tuyến tính của SHAP.
- Với Isotonic regression, đây là hàm bậc thang phi tham số nên đạt Brier score rất thấp nhưng làm mất tính cộng tính tuyến tính cho SHAP.

In [ ]:
cal_results = {}
rows = []
cv_rows = []

for name, pipe in models.items():
    comp = compare_calibrations_on_valid(pipe, X_val, y_val, cost_fn=0.45, cost_fp=0.05, ead_for_cv=ead_val)
    cal_results[name] = comp
    a_val = comp["platt_params"]["a"]
    b_val = comp["platt_params"]["b"]
    for method in ["uncalibrated", "platt", "isotonic"]:
        m = comp["methods"][method]["metrics"]
        cv_m = comp["cv_methods"][method]["metrics"]
        cv_rows.append({
            "Model": name,
            "Method": method.capitalize(),
            "OOF Brier": cv_m["brier_score"],
            "OOF Log Loss": cv_m["log_loss"],
            "OOF ROC-AUC": cv_m["roc_auc"],
            "OOF ECE": cv_m["ece"],
        })
        rows.append({
            "Mô hình": name,
            "Phương pháp": method.capitalize(),
            "Brier Score": m["brier_score"],
            "Log Loss": m["log_loss"],
            "ROC-AUC": m["roc_auc"],
            "ECE": m["ece"],
            "Platt a": a_val if method == "platt" else np.nan,
            "Platt b": b_val if method == "platt" else np.nan,
        })

cal_df = pd.DataFrame(rows)
display(cal_df)

print("5-fold OOF metrics (each calibrator is fit without its held-out fold):")
display(pd.DataFrame(cv_rows))

cv_cost_rows = []
for name, comp in cal_results.items():
    for method in ["platt", "isotonic"]:
        cv_cost = comp["cv_cost"][method]
        cv_cost_rows.append({
            "Model": name,
            "Method": method.capitalize(),
            "Mean threshold +/- SD": f"{cv_cost['threshold_mean']:.3f} +/- {cv_cost['threshold_std']:.3f}",
            "Mean held-out cost +/- SD": f"{cv_cost['heldout_cost_mean']:.5f} +/- {cv_cost['heldout_cost_std']:.5f}",
        })
print("Threshold selected on each training fold; cost measured on its held-out fold (unit cost):")
display(pd.DataFrame(cv_cost_rows))

ead_cv_rows = []
for name, comp in cal_results.items():
    for method in ["platt", "isotonic"]:
        cv_cost = comp["cv_cost_ead"][method]
        ead_cv_rows.append({
            "Model": name,
            "Method": method.capitalize(),
            "Mean threshold +/- SD": f"{cv_cost['threshold_mean']:.3f} +/- {cv_cost['threshold_std']:.3f}",
            "Mean held-out cost in NT$ per customer +/- SD": f"{cv_cost['heldout_cost_mean']:.2f} +/- {cv_cost['heldout_cost_std']:.2f}",
        })
print("EAD-weighted CV cost (currency unit: NT$):")
display(pd.DataFrame(ead_cv_rows))

## 3. Biểu đồ Đường cong Hiệu chuẩn (Reliability Diagrams)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, comp) in zip(axes, cal_results.items()):
    ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration", alpha=0.7)
    for m_name, style, col in [("uncalibrated", "s-", "#e74c3c"), ("platt", "o-", "#2ecc71"), ("isotonic", "^-", "#3498db")]:
        m = comp["methods"][m_name]["metrics"]
        ax.plot(m["prob_pred"], m["prob_true"], style, label=f"{m_name.capitalize()} (Brier={m['brier_score']:.4f})", color=col, linewidth=2)
    ax.set_title(name, fontsize=12, fontweight="bold")
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Fraction of Positives")
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 4. Tối ưu hóa Ngưỡng theo Ma trận Chi phí

Theo Project Charter mục 1.3:
- Chi phí FN = LGD × EAD = 0.45 (bỏ sót khách vỡ nợ).
- Chi phí FP = 5% × EAD = 0.05 (giảm oan hạn mức khách tốt).
- Tỷ lệ chi phí chuẩn FN:FP = 9:1.

In [ ]:
thresholds = np.linspace(0.02, 0.98, 97)
plt.figure(figsize=(9, 5))
for name, comp in cal_results.items():
    p = comp["methods"]["platt"]["y_prob"]
    costs = [compute_expected_cost(y_val, p, threshold=th, cost_fn=0.45, cost_fp=0.05, normalize=True) for th in thresholds]
    opt_th = comp["methods"]["platt"]["optimal_threshold"]
    min_c = comp["methods"]["platt"]["min_expected_cost"]
    plt.plot(thresholds, costs, label=f"{name} (Opt Th={opt_th:.2f}, Cost={min_c:.4f})", linewidth=2)
    plt.scatter([opt_th], [min_c], s=70, zorder=5)

plt.axvline(0.5, color="gray", linestyle="--", label="Ngưỡng mặc định 0.50")
plt.title("Đường cong Chi phí Kỳ vọng theo Ngưỡng Quyết định (Platt Calibrated)", fontweight="bold")
plt.xlabel("Ngưỡng Quyết định (Probability Threshold)")
plt.ylabel("Chi phí Kỳ vọng Đơn vị (Expected Cost per Customer)")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.show()

cost_rows = []
for name, comp in cal_results.items():
    m = comp["methods"]["platt"]
    cost_rows.append({
        "Mô hình": name,
        "Ngưỡng Tối ưu": m["optimal_threshold"],
        "Chi phí tại Ngưỡng Tối ưu": m["min_expected_cost"],
        "Chi phí tại Ngưỡng 0.50": m["cost_at_05"],
        "Mức giảm chi phí (%)": (m["cost_at_05"] - m["min_expected_cost"]) / m["cost_at_05"] * 100,
        "Recall tại Ngưỡng Tối ưu": m["cm_at_optimal"]["recall"],
        "Flag Rate tại Ngưỡng Tối ưu": m["cm_at_optimal"]["flag_rate"],
        "Precision tại Ngưỡng Tối ưu": m["cm_at_optimal"]["precision"],
    })
display(pd.DataFrame(cost_rows))

## 5. Phân tích Độ nhạy theo Tỷ lệ Chi phí FN:FP (3:1 đến 10:1)

In [ ]:
lgb_comp = cal_results["LightGBM tuned + Monotonic"]
p_lgb = lgb_comp["methods"]["platt"]["y_prob"]
sens_res = sensitivity_analysis_cost_ratios(y_val, p_lgb, cost_ratios=(3.0, 5.0, 7.0, 9.0, 10.0))
sens_df = pd.DataFrame(sens_res)
display(sens_df)

## 6. Quy đổi PD sang Điểm Tín dụng theo PDO (Thang 300–850)

Công thức PDO: $\text{Score} = \text{Offset} + \text{Factor} \cdot \ln(\frac{1-\text{PD}}{\text{PD}})$.
Với Base Score = 600 tại odds 50:1 và PDO = 20: Factor ≈ 28.854, Offset ≈ 487.123.

In [ ]:
cal_pipe = CalibratedPipeline(models["LightGBM tuned + Monotonic"], method="platt")
cal_pipe.fit_calibrator(X_val, y_val, optimize_thresh=True)
scores = cal_pipe.predict_score(X_val)
cutoff_score = pd_to_score(cal_pipe.optimal_threshold_, clip=(300, 850))

plt.figure(figsize=(9, 5))
plt.hist(scores, bins=40, color="#3498db", edgecolor="white", alpha=0.85, label="Hồ sơ Valid")
plt.axvline(cutoff_score, color="#e74c3c", linestyle="--", linewidth=2.5, label=f"Cutoff Ngưỡng Tối ưu: {cutoff_score:.1f} điểm")
plt.title("Phân phối Điểm Tín dụng PDO (LightGBM tuned + Monotonic) trên Valid", fontweight="bold")
plt.xlabel("Điểm tín dụng")
plt.ylabel("Số lượng khách hàng")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.show()

print("Thống kê Điểm tín dụng Valid:")
print(f"- Điểm trung bình: {scores.mean():.1f} ± {scores.std():.1f}")
print(f"- Trung vị: {np.median(scores):.1f}")
print(f"- Điểm thấp nhất: {scores.min():.1f}, Điểm cao nhất: {scores.max():.1f}")
print(f"- Cutoff phân loại tại ngưỡng tối ưu (p*={cal_pipe.optimal_threshold_:.2f}): {cutoff_score:.1f} điểm")
print(f"  + Khách hàng có Điểm <= {cutoff_score:.1f} (PD >= {cal_pipe.optimal_threshold_:.2f}): Gắn cờ Cảnh báo / Can thiệp ({np.mean(scores <= cutoff_score)*100:.1f}%)")
print(f"  + Khách hàng có Điểm > {cutoff_score:.1f} (PD < {cal_pipe.optimal_threshold_:.2f}): Đạt chuẩn Bình thường ({np.mean(scores > cutoff_score)*100:.1f}%)")

## 7. Kết luận & Khuyến nghị Kỹ thuật cho Tuần 3

1. Bảng fit/đo trên cùng Valid chỉ mang tính tham khảo; quyết định Platt hay Isotonic cần dựa trên bảng OOF 5-fold ở trên. Platt giữ được quan hệ tuyến tính với margin để diễn giải SHAP, nhưng lợi ích calibration phải được xác nhận ngoài mẫu.
2. Ngưỡng và chi phí tối ưu trên toàn Valid là in-sample. Dùng kết quả `cv_cost` theo fold để đánh giá quy trình chọn ngưỡng; chưa coi mức giảm chi phí hoặc `p*=0.11` là kết quả triển khai. Vùng 0.09–0.13 là vùng ứng viên theo đường chi phí hiện tại, cần kiểm tra lại trên CV/Test.
3. Điểm trung bình 534.2 không quy đổi trực tiếp thành PD trung bình do hàm PDO phi tuyến. Ngưỡng điểm 547.5 hiện là cutoff ứng với ngưỡng in-sample 0.11.
4. Ngưỡng 0.11 gắn cờ khoảng 64.7% hồ sơ và precision khoảng 30.5%; cần đối chiếu năng lực xử lý cảnh báo.
5. Chi phí EAD được tính theo BILL_AMT1, đơn vị tiền tệ gốc NT$ (TWD). Chỉ chạy nghiệm thu Test sau khi khóa quy trình chọn calibrator/ngưỡng trên Valid.